In [ ]:
# Setup: Import modules and define paths
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Now import validation functions
from functions.validation_functions import (
    load_groundwater_csv,
    remove_duplicate_and_fill_missing,
    flag_physical_bounds,
    flag_unrealistic_step_change,
    flag_constant_head_periods,
    flag_statistical_outliers,
)

# Dataset roots (per-origin structure)
wiertsema_input_dir = repo_root / 'output_data' / 'wiertsema'
fugro_input_dir = repo_root / 'output_data' / 'fugro'
output_dir = repo_root / 'output_data' / 'single_validation'

output_dir.mkdir(parents=True, exist_ok=True)

print('Setup complete!')
print(f'Repo root: {repo_root}')
print(f'Wiertsema dataset root: {wiertsema_input_dir}')
print(f'Fugro dataset root: {fugro_input_dir}')
print(f'Output directory: {output_dir}')

# Single File Processing (Interactive)

In [ ]:
# Step 2: Specify which file to analyze
dataset_choice = 'wiertsema'  # 'wiertsema' or 'fugro'
file_to_analyze = '86349-1 HB002PB01 HB_BE0072+3_BIKR_GMW_PB1_F-246.csv'

dataset_root = wiertsema_input_dir if dataset_choice.lower() == 'wiertsema' else fugro_input_dir
matches = sorted(dataset_root.glob(f'*/knmi/{file_to_analyze}'))
if not matches:
    raise FileNotFoundError(f"Could not find {file_to_analyze} under {dataset_root}/<origin>/knmi")
selected_file = matches[0]

# Step 3: Load the CSV file using imported function
print(f'\nLoading: {selected_file.name}')
df = load_groundwater_csv(selected_file)
print(f'Loaded {len(df)} rows')
print(f'  Time range: {df["Time"].min()} to {df["Time"].max()}')
print(f'\nFirst 5 rows:')
print(df.head())

# Setting parameters

In [27]:
# Step 1: Define validation parameters (adjust these based on your data)
# Physical bounds (meters NAP or similar datum)
hmin = -10  # Minimum realistic head value
hmax = 10   # Maximum realistic head value

# Maximum allowed rate of change between timestamps in m/day
max_up = 0.15
max_down = -0.05  

# Constant head periods
tconst_steps = 72   # Flag if at least 24 hourly measurements are constant
flat_margin_m = 0.02  # Checking the 'dH' column for changes smaller than this
min_band_m = 0.15   # Using the minimum value from the head series, then adding this margin on top


print('Validation parameters set:')
print(f'  Physical bounds: {hmin} to {hmax} m')
#print(f'  Constant head: >{tconst_days} days or >{nconst_min} measurements')

Validation parameters set:
  Physical bounds: -10 to 10 m


### Step 1: Removing double timestamps

In [8]:
# Step 1 in the pipeline: remove duplicate timestamps
df_removed_double_timestamps, had_dupes = remove_duplicate_and_fill_missing(df)

# Printing results
print(f"\n Before duplicate removal: {len(df)} After duplicate removal: {len(df_removed_double_timestamps)} rows")
print(f"Any duplicates removed? {had_dupes}")


 Before duplicate removal: 11279 After duplicate removal: 11353 rows
Any duplicates removed? True


d:\Users\jvanruitenbeek\data_validation\functions\validation_functions.py:123: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_missing], ignore_index=True)


### Step 1b: Copying the original head series for checking

In [9]:
df_removed_double_timestamps["head_og"] = df_removed_double_timestamps["head"].copy()

### Step 2: Flag Physical bounds

In [10]:
df_v1, had_bounds = flag_physical_bounds(df_removed_double_timestamps, hmin, hmax)

# Printing results
print(f"\nBefore flagging: {len(df_removed_double_timestamps)} After flagging physical bounds: {len(df_v1)} rows")
print(f"Any physical bounds flagged? {had_bounds}")


Before flagging: 11353 After flagging physical bounds: 11353 rows
Any physical bounds flagged? False


d:\Users\jvanruitenbeek\data_validation\functions\validation_functions.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df_out.loc[mask_flagged & df_out[vcol].isna(), vcol] = df_out.loc[mask_flagged, "head"]


### Step 3: Flag unrealistic_step_changes

In [11]:
df_v2, had_steps = flag_unrealistic_step_change(df_v1, max_up, max_down)

In [12]:
df_v2['head'].min

<bound method Series.min of 0       -3.051
1       -3.049
2       -3.047
3       -3.046
4       -3.043
         ...  
11348   -2.945
11349   -2.948
11350   -2.953
11351   -2.955
11352   -2.950
Name: head, Length: 11353, dtype: float64>

### Step 4: Check for constant head (droogstand)

In [21]:
test_value = -2.81

In [22]:
df_v3, had_droogstand = flag_constant_head_periods(df_v2, tconst_steps, flat_margin_m, min_band_m, test_value)

hmin used = -2.81
Threshold detection value = -2.66


### Step 5: Check for outliers based on 1.5 IQR and -1.5 IQR

In [23]:
df_v4, had_outliers = flag_statistical_outliers(df_v3)

In [24]:
df_final = df_v4.drop(columns=["dH"])

In [25]:
df_final.info

<bound method DataFrame.info of                      Time  head    v1  head_og  v2     v3  v4
0     2024-08-14 12:00:00   NaN  True   -3.051 NaN -3.051 NaN
1     2024-08-14 13:00:00   NaN  True   -3.049 NaN -3.049 NaN
2     2024-08-14 14:00:00   NaN  True   -3.047 NaN -3.047 NaN
3     2024-08-14 15:00:00   NaN  True   -3.046 NaN -3.046 NaN
4     2024-08-14 16:00:00   NaN  True   -3.043 NaN -3.043 NaN
...                   ...   ...   ...      ...  ..    ...  ..
11348 2025-11-29 20:00:00   NaN  True   -2.945 NaN -2.945 NaN
11349 2025-11-29 21:00:00   NaN  True   -2.948 NaN -2.948 NaN
11350 2025-11-29 22:00:00   NaN  True   -2.953 NaN -2.953 NaN
11351 2025-11-29 23:00:00   NaN  True   -2.955 NaN -2.955 NaN
11352 2025-11-30 00:00:00   NaN  True   -2.950 NaN -2.950 NaN

[11353 rows x 7 columns]>

### Creating a plot

In [26]:
import plotly.graph_objects as go

# Ensure Time is datetime (safe even if it already is)
df_final["Time"] = pd.to_datetime(df_final["Time"])

fig = go.Figure()

# Main head time series (black line)
fig.add_trace(
    go.Scatter(
        x=df_final["Time"],
        y=df_final["head"],
        mode="lines",
        name="Head",
        line=dict(color="black", width=2),
        connectgaps=False,
    )
)

# Helper to only plot non-NaN points for v-columns
def add_flag_trace(df, col, name, color):
    mask = df[col].notna()
    if mask.any():
        fig.add_trace(
            go.Scatter(
                x=df.loc[mask, "Time"],
                y=df.loc[mask, col],
                mode="markers",
                name=name,
                marker=dict(color=color, size=7),
            )
        )

# v1, v2, v3 as different reds
add_flag_trace(df_final, "v1", "v1 (physical bounds)", "red")
add_flag_trace(df_final, "v2", "v2 (step change)", "darkred")
add_flag_trace(df_final, "v3", "v3 (constant head)", "tomato")

# NEW: v4 outliers (use green)
add_flag_trace(df_final, "v4", "v4 (statistical outliers)", "green")

fig.update_layout(
    title="Groundwater Head with Validation Flags",
    xaxis_title="Time",
    yaxis_title="Head [m]",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()